# 2025 전력사용량 예측 AI 경진대회 Review
## **코드 흐름 요약**
파이프라인: 데이터 전처리 -> 도메인 기반 파생변수 생성 -> 노이즈/이상치 제거 -> 통계량 변수 병합 -> 변수 최적화 및 모델 앙상블.

* 파생변수 생성 (Feature Engineering): 시간/달력: 요일, 주말, 공휴일, 여름철 주기성(cos/sin) 반영.
  * 기온/기상: 4일·7일 이동평균, 일교차, 쾌적도 범주, 강수 전후 인디케이터.
  * 핵심: 누적 냉방부하(CDH) 및 불쾌지수(THI). 일정 온도(26도)를 넘은 상태가 과거 몇 시간 동안 지속되었는지 누적치를 계산해 냉방 수요 강도를 수치화.

* 이상치 및 휴일 수동 매핑 (Data Cleansing):
  * 모델에 혼란을 주는 불규칙한 건물별 휴일과, 정전 등으로 추정되는 일시적 전력 스파이크(이상치)를 직접 찾아내어 선형 보간하거나 아예 제거.

* 통계량 변수 산출 (Statistical Features):
  * 휴일 여부, 시간대별, 월별 전력 소비량의 평균과 표준편차를 계산. (이때 Validation 데이터가 오염되지 않도록 시점 분리에 매우 주의함).

* 건물별 맞춤 변수 탐색 (Feature Selection):
  * 전체 변수 중 각 건물에 오히려 악영향을 주는 노이즈 변수를 찾기 위해, 하나씩 변수를 빼가며 성능(SMAPE)을 평가하는 후진 대입법(Backward Elimination) 수행.

* 모델 학습 및 앙상블 (Modeling & Seed Ensemble):

  * 작은 데이터셋에 최적화된 사전학습 파운데이션 모델 TabPFN 활용.
  * 데이터가 적어 생기는 모델의 편향(Variance)을 줄이기 위해, 랜덤 시드(Seed)를 100개 바꿔가며 동일 모델을 100번 학습시키고 그 평균값으로 최종 예측.

## **새롭게 알게 된 내용 및 배울 점**
1억 개 이상의 합성 데이터로 사전학습된 트랜스포머 모델을 사용하여, 샘플 수가 적고 변수가 많은 환경(건물당 약 2,000건 남짓)에서 발생할 수 있는 과적합을 막아낼 수 있음을 새롭게 알 수 있었다.

또한 단순히 현재 온도만 입력하는 것을 넘어, 기준 온도(26도)를 초과한 값의 최근 12시간 누적합을 구하는 도메인 지식 기반의 피처 엔지니어링 기술을 배웠다.

복잡한 서로 다른 모델을 섞는 앙상블 대신, 가장 성능이 좋은 동일 모델의 Seed만 변경하여 무려 100번 반복 학습시킨 뒤 평균을 내어 분산(Variance)을 철저하게 잡아낸 것도 배울 점이라 생각한다.

## **코드**

In [ ]:
from tabpfn import TabPFNRegressor

# 사전학습 데이터 제한을 무시하고 GPU 위에서 훈련을 수행하도록 설정
model = TabPFNRegressor(random_state=seed, device=device, ignore_pretraining_limits=True)
model.fit(X_train_np, y_train_np)
y_preds.append(np.asarray(model.predict(X_test_np), dtype=np.float32))

In [ ]:
for feat in candidates:
    # 현재 타겟이 된 변수(feat) 1개만 제외하고 학습/검증 데이터를 구성
    trial_feats = [f for f in X_all if f != feat]
    X_tr = X_train_full[trial_feats]; X_vl = X_val_full[trial_feats]

    # 모델 학습 후 에러율(SMAPE) 계산
    trial_smape, _ = tabpfn_fit_predict(X_tr, y_train, X_vl, y_val, device=device)

    # 기존보다 성능이 개선되었다면(에러가 줄었다면) 이 변수를 제거 후보로 지정
    if (baseline_smape - trial_smape >= tol) and (trial_smape < best_smape):
        best_smape = trial_smape
        best_feat = feat

In [ ]:
y_preds = []
# 랜덤 시드만 0부터 99까지 변경하며 100번의 독립적인 모델 학습/예측 수행
for seed in range(n_ensembles):
    model = TabPFNRegressor(random_state=seed, device=device, ignore_pretraining_limits=True)
    model.fit(X_train_np, y_train_np)
    y_preds.append(np.asarray(model.predict(X_test_np), dtype=np.float32))

# 100개의 예측값을 평균 내어 안정적이고 강건한(Robust) 최종 결과 도출
y_test_pred = np.mean(y_preds, axis=0).astype(np.float32)